# XGBoost Taxi Destination Notebook

Standard three-mode XGBoost workflow: validation-ranked hybrid tuning,
selected-model validation diagnostics, and final round calibration plus
complete-data fit/submission.


## Cached Data Paths

Load the shared cached preprocessing dataset and write standard Kaggle outputs.


In [ ]:
import os
from pathlib import Path

PREPROCESSED_DATA_DIR = Path(
    os.environ.get(
        "PREPROCESSED_DATA_DIR",
        "/kaggle/input/datasets/louishogge/preprocessed-taxi-dataset/data",
    )
)
OUTPUT_DIR = Path("/kaggle/working")
SUBMISSION_DIR = OUTPUT_DIR / "submission"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
ENSEMBLE_EXPORT_DIR = OUTPUT_DIR / "ensemble_inputs"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessed data directory:", PREPROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR.resolve())
print("Submission directory:", SUBMISSION_DIR.resolve())
print("Artifact directory:", ARTIFACT_DIR.resolve())
print("Ensemble export directory:", ENSEMBLE_EXPORT_DIR.resolve())


## Imports

Import the deterministic training, evaluation, plotting, and serialization stack.


In [ ]:
import gc
import json
import random
import time
from copy import deepcopy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from packaging.version import Version
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor


## Hyperparameters

Enable at most one expensive mode. Tuning creates unreviewed in-memory evidence;
validation refit and final fit use the reviewed target-specific configuration. Set `RUN_VALIDATION_REFIT=True` and `RUN_FINAL_FIT=False` to regenerate the reviewed model's aligned ensemble-prediction Parquet and provenance sidecar under `/kaggle/working/ensemble_inputs`.


In [ ]:
SEED = 42
MODEL_NAME = "XGBoost"
MODEL_SLUG = "xgboost"

RUN_HYPERPARAMETER_TUNING = False
RUN_VALIDATION_REFIT = False
RUN_FINAL_FIT = True

MAX_BOOST_ROUNDS = 10_000
EARLY_STOPPING_ROUNDS = 100
SEARCH_STOP_FRACTION = 0.10
FINAL_ROUND_STOP_FRACTION = 0.10
FINAL_ROUND_CALIBRATION_SEED = 42
MIN_XGBOOST_VERSION = Version("3.2.0")

WANDB_PROJECT = "machine-learning"
WANDB_ENTITY = "ml-l"
WANDB_SECRET_NAME = "wandb-api-key"

TARGET_COLUMNS = ["END_LONG", "END_LAT"]
TARGET_SPECS = [
    ("END_LONG", "longitude"),
    ("END_LAT", "latitude"),
]
TARGET_TO_PAYLOAD = {
    "END_LONG": "longitude",
    "END_LAT": "latitude",
}
GROUP_SUFFIX_SEPARATOR = "__prefix_"

FIXED_MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "booster": "gbtree",
    "tree_method": "hist",
    "device": "cuda",
    "eval_metric": "rmse",
    "grow_policy": "depthwise",
    "max_bin": 256,
    "gamma": 0.0,
    "sampling_method": "uniform",
    "random_state": SEED,
    "n_jobs": -1,
}

BEST_TARGET_MODEL_PARAMS = {
    "END_LONG": {
        "learning_rate": 0.05,
        "max_depth": 14,
        "min_child_weight": 50,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0,
    },
    "END_LAT": {
        "learning_rate": 0.05,
        "max_depth": 14,
        "min_child_weight": 25,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 2.5,
        "reg_alpha": 0.0,
    },
}
BEST_TARGET_N_ESTIMATORS = {
    "END_LONG": 766,
    "END_LAT": 312,
}
BEST_MODEL_RUN_SUMMARY = {
    "reviewed_on": "2026-07-13",
    "source": "Completed W&B validation-ranked search and hybrid-results table",
    "tuner_type": "canonical_summary",
    "selection_status": "reviewed_hybrid_validation_refit_completed",
    "source_wandb_run": "xgboost-search-summary",
    "source_run_created_at": "2026-07-13T19:23:36.000Z",
    "source_hybrid_rank": 1,
    "train_haversine_mean_km": 1.8527836908535569,
    "train_haversine_median_km": 1.1708611039486865,
    "train_haversine_std_km": 2.3714743264220943,
    "train_haversine_p90_km": 3.992051908359309,
    "validation_haversine_mean_km": 2.214959443795627,
    "validation_haversine_median_km": 1.2951824933837817,
    "validation_haversine_std_km": 3.3366981385498717,
    "validation_haversine_p90_km": 4.6953666568030785,
    "generalization_gap_km": 0.36217575294206994,
    "fit_seconds": 130.3868911266327,
    "local_submission_rank_by_public_mean": 3,
    "local_submission_all_mean_km": 2.401692751943281,
    "local_submission_all_median_km": 1.348754570240231,
    "local_submission_all_std_km": 3.0514497583019256,
    "local_submission_all_p90_km": 5.647785129089402,
    "local_submission_public_mean_km": 2.276327581622824,
    "local_submission_private_mean_km": 2.527057922263739,
    "local_submission_n_trips": 320,
    "validation_refit_fit_seconds": 130.3868911266327,
    "xgboost_version": "3.2.0",
    "search_completed": True,
    "selected_model_validation_refit_completed": True,
    "final_fit_completed": False,
    "target_results": {
        "END_LONG": {
            "target_column": "END_LONG",
            "target_label": "longitude",
            "candidate_index": 30,
            "candidate_label": "micro_05_depth14_moderated",
            "selected_n_estimators": 766,
            "best_iteration": 765,
            "best_inner_stop_rmse": 0.037405483,
            "validation_rmse": 0.0333995444018782,
            "early_stopping_fit_seconds": 78.80079674720764,
            "validation_refit_fit_seconds": 78.70022583007812,
        },
        "END_LAT": {
            "target_column": "END_LAT",
            "target_label": "latitude",
            "candidate_index": 29,
            "candidate_label": "micro_04_depth14_less_regularized",
            "selected_n_estimators": 312,
            "best_iteration": 311,
            "best_inner_stop_rmse": 0.02523526,
            "validation_rmse": 0.0258023355237222,
            "early_stopping_fit_seconds": 55.15665674209595,
            "validation_refit_fit_seconds": 51.686665296554565,
        },
    },
}

SUBMISSION_FILE = f"submission_{MODEL_SLUG}.csv"
BEST_PARAMS_PATH = ARTIFACT_DIR / f"best_params_{MODEL_SLUG}.json"
MODEL_BUNDLE_PATH = ARTIFACT_DIR / f"model_bundle_{MODEL_SLUG}.joblib"
VALIDATION_PREDICTIONS_PATH = ENSEMBLE_EXPORT_DIR / f"validation_predictions_{MODEL_SLUG}.parquet"
VALIDATION_PREDICTIONS_MANIFEST_PATH = ENSEMBLE_EXPORT_DIR / f"validation_predictions_{MODEL_SLUG}.manifest.json"
VALIDATION_EXPORT_DRIFT_TOLERANCE_KM = 0.001

enabled_expensive_stages = sum(bool(value) for value in (
    RUN_HYPERPARAMETER_TUNING,
    RUN_VALIDATION_REFIT,
    RUN_FINAL_FIT,
))
if enabled_expensive_stages > 1:
    raise ValueError(
        "Enable only one of RUN_HYPERPARAMETER_TUNING, RUN_VALIDATION_REFIT, "
        "or RUN_FINAL_FIT for a Kaggle run."
    )


def require_selected_model_configuration():
    """Validate the reviewed target-specific parameters and round counts."""
    if set(BEST_TARGET_MODEL_PARAMS) != set(TARGET_COLUMNS):
        raise ValueError("BEST_TARGET_MODEL_PARAMS must cover both targets.")
    if set(BEST_TARGET_N_ESTIMATORS) != set(TARGET_COLUMNS):
        raise ValueError("BEST_TARGET_N_ESTIMATORS must cover both targets.")
    for target_column in TARGET_COLUMNS:
        params = BEST_TARGET_MODEL_PARAMS[target_column]
        rounds = BEST_TARGET_N_ESTIMATORS[target_column]
        if not isinstance(params, dict) or not params:
            raise ValueError(f"Missing reviewed parameters for {target_column}.")
        if not isinstance(rounds, int) or isinstance(rounds, bool) or rounds <= 0:
            raise ValueError(f"Missing positive reviewed rounds for {target_column}.")
    if RUN_FINAL_FIT and not BEST_MODEL_RUN_SUMMARY.get(
        "selected_model_validation_refit_completed",
        False,
    ):
        raise ValueError(
            "Run and review the selected-model validation refit before final fit."
        )


if RUN_VALIDATION_REFIT or RUN_FINAL_FIT:
    require_selected_model_configuration()


## Reproducibility


In [ ]:
def seed_everything(seed=SEED):
    """Seed deterministic Python and NumPy behavior."""
    random.seed(seed)
    np.random.seed(seed)


seed_everything()


## CUDA Requirement

Expensive XGBoost modes require a visible CUDA device and never fall back to CPU.


In [ ]:
def validate_xgboost_and_cuda_runtime():
    """Validate the XGBoost version and require CUDA for expensive modes."""
    installed_version = Version(xgb.__version__)
    if installed_version < MIN_XGBOOST_VERSION:
        raise RuntimeError(
            f"XGBoost {MIN_XGBOOST_VERSION} or newer is required; "
            f"found {installed_version}."
        )

    runtime = {
        "xgboost_version": str(installed_version),
        "cuda_checked": False,
        "cuda_device_count": None,
    }
    if enabled_expensive_stages == 0:
        return runtime

    try:
        import cupy as cp

        device_count = int(cp.cuda.runtime.getDeviceCount())
    except Exception as exception:
        raise RuntimeError(
            "A Kaggle CUDA accelerator is required. Enable GPU acceleration "
            "before running an expensive XGBoost stage."
        ) from exception

    if device_count < 1:
        raise RuntimeError("No CUDA device is visible; CUDA is required.")

    runtime["cuda_checked"] = True
    runtime["cuda_device_count"] = device_count
    return runtime


XGBOOST_RUNTIME = validate_xgboost_and_cuda_runtime()
print("XGBoost version:", XGBOOST_RUNTIME["xgboost_version"])
if XGBOOST_RUNTIME["cuda_checked"]:
    print("Visible CUDA devices:", XGBOOST_RUNTIME["cuda_device_count"])


## Utility Constants


In [ ]:
PREPROCESSED_SCHEMA_VERSION = 1
EXPECTED_CATEGORICAL_ENCODING_POLICY = {
    "CALL_TYPE": "one_hot",
    "ORIGIN_CALL": "missing_indicator_plus_log_count",
    "ORIGIN_STAND": "one_hot_missing_as_0",
    "TAXI_ID": "one_hot",
}


## Cached Preprocessed Dataset Helpers


In [ ]:
def require_existing_path(path, description):
    """Return an existing path or fail with an actionable Kaggle message."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}. Attach the cached Kaggle Dataset "
            "or set PREPROCESSED_DATA_DIR."
        )
    return path


def load_preprocessed_metadata(base_dir=PREPROCESSED_DATA_DIR):
    """Load and validate cached preprocessing metadata."""
    metadata_path = require_existing_path(
        base_dir / "metadata.json",
        "preprocessed metadata",
    )
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    if metadata.get("schema_version") != PREPROCESSED_SCHEMA_VERSION:
        raise ValueError("The attached preprocessing schema version is unsupported.")
    policy = metadata.get("constants", {}).get("categorical_encoding_policy")
    if policy != EXPECTED_CATEGORICAL_ENCODING_POLICY:
        raise ValueError("The attached categorical encoding policy does not match.")
    return metadata


def load_cached_frame(relative_path):
    """Load one cached Parquet frame."""
    frame_path = require_existing_path(
        PREPROCESSED_DATA_DIR / relative_path,
        f"cached frame {relative_path}",
    )
    return pd.read_parquet(frame_path)


def assert_index_aligned(features, targets, split_name):
    """Require exact feature/target row alignment."""
    if not features.index.equals(targets.index):
        raise ValueError(f"Cached {split_name} features and targets are misaligned.")


def assert_matching_features(left, right, left_name, right_name):
    """Require identical ordered feature columns."""
    if not left.columns.equals(right.columns):
        raise ValueError(
            f"Feature columns differ between {left_name} and {right_name}."
        )


def assert_target_columns(targets, split_name):
    """Require target columns in cached [LONG, LAT] order."""
    if list(targets.columns) != TARGET_COLUMNS:
        raise ValueError(
            f"Cached {split_name} target columns are {list(targets.columns)!r}; "
            f"expected {TARGET_COLUMNS!r}."
        )


def extract_trip_group(index_value, separator=GROUP_SUFFIX_SEPARATOR):
    """Recover the original trip identifier from one cached prefix index."""
    value = str(index_value)
    if separator not in value:
        raise ValueError(
            f"Cached index {value!r} does not contain separator {separator!r}."
        )
    return value.rsplit(separator, 1)[0]


def make_group_disjoint_split(index, stop_fraction, seed):
    '''Return deterministic train/stop positions with disjoint trips.'''
    if not 0.0 < float(stop_fraction) < 1.0:
        raise ValueError("stop_fraction must be strictly between zero and one.")
    groups = np.asarray(
        [extract_trip_group(value) for value in index],
        dtype=object,
    )
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=float(stop_fraction),
        random_state=int(seed),
    )
    train_positions, stop_positions = next(
        splitter.split(np.zeros(len(index), dtype=np.uint8), groups=groups)
    )
    train_groups = set(groups[train_positions])
    stop_groups = set(groups[stop_positions])
    if train_groups.intersection(stop_groups):
        raise AssertionError("Original trip groups overlap across partitions.")
    return train_positions, stop_positions, groups


def load_cached_validation_data():
    """Load canonical validation train and holdout matrices."""
    X_train = load_cached_frame("validation/X_train.parquet")
    y_train = load_cached_frame("validation/y_train.parquet")
    X_val = load_cached_frame("validation/X_val.parquet")
    y_val = load_cached_frame("validation/y_val.parquet")
    assert_index_aligned(X_train, y_train, "validation train")
    assert_index_aligned(X_val, y_val, "validation holdout")
    assert_matching_features(X_train, X_val, "X_train", "X_val")
    assert_target_columns(y_train, "validation train")
    assert_target_columns(y_val, "validation holdout")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    return X_train, y_train, X_val, y_val


def load_cached_final_data():
    """Load cached full-train and competition-test matrices."""
    final_dir = require_existing_path(
        PREPROCESSED_DATA_DIR / "final",
        "final data directory",
    )
    scaler_path = require_existing_path(
        final_dir / "gps_scaler.joblib",
        "final GPS scaler",
    )
    encoder_path = require_existing_path(
        final_dir / "categorical_encoder.joblib",
        "final categorical encoder",
    )

    X_final_train = load_cached_frame("final/X_train.parquet")
    y_final_train = load_cached_frame("final/y_train.parquet")
    X_final_test = load_cached_frame("final/X_test.parquet")
    final_gps_scaler = joblib.load(scaler_path)
    final_categorical_encoder = joblib.load(encoder_path)

    assert_index_aligned(X_final_train, y_final_train, "final train")
    assert_matching_features(
        X_final_train,
        X_final_test,
        "X_final_train",
        "X_final_test",
    )
    assert_target_columns(y_final_train, "final train")
    print("X_final_train shape:", X_final_train.shape)
    print("y_final_train shape:", y_final_train.shape)
    print("X_final_test shape:", X_final_test.shape)
    return (
        X_final_train,
        y_final_train,
        X_final_test,
        final_gps_scaler,
        final_categorical_encoder,
    )


PREPROCESSED_METADATA = load_preprocessed_metadata()


## Evaluation And Submission Helpers


In [ ]:
def make_json_safe(value):
    """Convert notebook values to strict JSON-compatible objects."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())
    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value


def long_lat_to_lat_long(values):
    """Convert a two-column [LONG, LAT] array to [LAT, LONG]."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != 2:
        raise ValueError(f"Expected [LONG, LAT] values; got shape {values.shape}.")
    return np.column_stack([values[:, 1], values[:, 0]])


def combine_target_predictions(predictions_by_target):
    """Combine independent target predictions in [LONG, LAT] order."""
    missing = [target for target in TARGET_COLUMNS if target not in predictions_by_target]
    if missing:
        raise ValueError("Incomplete target predictions: " + ", ".join(missing))
    columns = [
        np.asarray(predictions_by_target[target], dtype=float).reshape(-1)
        for target in TARGET_COLUMNS
    ]
    if len({len(column) for column in columns}) != 1:
        raise ValueError("Longitude and latitude predictions have different lengths.")
    combined = np.column_stack(columns)
    if not np.isfinite(combined).all():
        raise ValueError("Combined predictions contain non-finite values.")
    return combined


def haversine_distances_km(y_true_long_lat, y_pred_long_lat):
    """Return row-wise Haversine errors for [LONG, LAT] arrays."""
    truth = long_lat_to_lat_long(y_true_long_lat)
    prediction = long_lat_to_lat_long(y_pred_long_lat)
    pred_lat = np.radians(prediction[:, 0])
    pred_long = np.radians(prediction[:, 1])
    true_lat = np.radians(truth[:, 0])
    true_long = np.radians(truth[:, 1])
    dlat = true_lat - pred_lat
    dlon = true_long - pred_long
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(pred_lat) * np.cos(true_lat) * np.sin(dlon / 2.0) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    return 2.0 * 6371.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def evaluate_long_lat_predictions(y_true_long_lat, y_pred_long_lat, prefix):
    """Calculate the standard Haversine metric set."""
    distances = haversine_distances_km(y_true_long_lat, y_pred_long_lat)
    return {
        f"{prefix}/haversine_mean_km": float(np.mean(distances)),
        f"{prefix}/haversine_median_km": float(np.median(distances)),
        f"{prefix}/haversine_std_km": float(np.std(distances)),
        f"{prefix}/haversine_p90_km": float(np.percentile(distances, 90)),
    }, distances


def rmse(y_true, y_pred):
    """Return scalar root mean squared error."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    prediction = np.asarray(y_pred, dtype=float).reshape(-1)
    return float(np.sqrt(np.mean((prediction - truth) ** 2)))


def write_submission(trip_ids, destinations, output_path):
    """Write one competition submission from [LAT, LONG] destinations."""
    destinations = np.asarray(destinations, dtype=float)
    expected_shape = (len(trip_ids), 2)
    if destinations.shape != expected_shape:
        raise ValueError(
            f"Expected destination shape {expected_shape}, got {destinations.shape}."
        )
    submission = pd.DataFrame(
        {
            "LATITUDE": destinations[:, 0],
            "LONGITUDE": destinations[:, 1],
        },
        index=trip_ids,
    )
    submission.to_csv(output_path, index_label="TRIP_ID")
    return submission


def save_json_artifact(payload, output_path):
    """Write one JSON artifact."""
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(
            make_json_safe(payload),
            file,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )


def reviewed_validation_metrics():
    """Return the reviewed metrics currently available in the summary."""
    summary = BEST_MODEL_RUN_SUMMARY
    summary_to_metric = {
        "train_haversine_mean_km": "train/haversine_mean_km",
        "train_haversine_median_km": "train/haversine_median_km",
        "train_haversine_std_km": "train/haversine_std_km",
        "train_haversine_p90_km": "train/haversine_p90_km",
        "validation_haversine_mean_km": "val/haversine_mean_km",
        "validation_haversine_median_km": "val/haversine_median_km",
        "validation_haversine_std_km": "val/haversine_std_km",
        "validation_haversine_p90_km": "val/haversine_p90_km",
        "generalization_gap_km": "generalization/haversine_gap_km",
    }
    metrics = {
        metric_key: summary[summary_key]
        for summary_key, metric_key in summary_to_metric.items()
        if summary.get(summary_key) is not None
    }
    metrics["validation_metrics_source"] = "reviewed_best_model_run_summary"
    return metrics


def save_best_params_artifact(
    validation_metrics,
    target_results,
    execution_status,
    final_target_n_estimators=None,
    final_round_calibration=None,
):
    '''Persist reviewed parameters, metrics, round provenance, and status.'''
    final_rounds = (
        None
        if final_target_n_estimators is None
        else {
            target: int(rounds)
            for target, rounds in final_target_n_estimators.items()
        }
    )
    run_summary = deepcopy(BEST_MODEL_RUN_SUMMARY)
    if execution_status.get("validation_refit_completed", False):
        run_summary.update({
            "selection_status": "reviewed_hybrid_validation_refit_completed",
            "selected_model_validation_refit_completed": True,
            "train_haversine_mean_km": validation_metrics[
                "train/haversine_mean_km"
            ],
            "train_haversine_median_km": validation_metrics[
                "train/haversine_median_km"
            ],
            "train_haversine_std_km": validation_metrics[
                "train/haversine_std_km"
            ],
            "train_haversine_p90_km": validation_metrics[
                "train/haversine_p90_km"
            ],
            "validation_haversine_mean_km": validation_metrics[
                "val/haversine_mean_km"
            ],
            "validation_haversine_median_km": validation_metrics[
                "val/haversine_median_km"
            ],
            "validation_haversine_std_km": validation_metrics[
                "val/haversine_std_km"
            ],
            "validation_haversine_p90_km": validation_metrics[
                "val/haversine_p90_km"
            ],
            "generalization_gap_km": validation_metrics[
                "generalization/haversine_gap_km"
            ],
        })
        for target_column in TARGET_COLUMNS:
            reviewed_target = run_summary["target_results"][target_column]
            run_summary["target_results"][target_column] = {
                **reviewed_target,
                **target_results[target_column],
            }
    payload = {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "params_source": "reviewed_target_specific_hybrid",
        "saved_target_model_params": BEST_TARGET_MODEL_PARAMS,
        "saved_target_n_estimators": BEST_TARGET_N_ESTIMATORS,
        "reviewed_target_n_estimators": BEST_TARGET_N_ESTIMATORS,
        "final_target_n_estimators": final_rounds,
        "final_round_calibration": final_round_calibration,
        "best_model_run_summary": run_summary,
        "fixed_model_params": FIXED_MODEL_PARAMS,
        "target_results": target_results,
        "validation_metrics": validation_metrics,
        "execution": execution_status,
        "xgboost_runtime": XGBOOST_RUNTIME,
        "preprocessed_dataset": {
            "path": str(PREPROCESSED_DATA_DIR),
            "schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "config": PREPROCESSED_METADATA.get("config", {}),
            "validation": PREPROCESSED_METADATA.get("validation", {}),
            "final": PREPROCESSED_METADATA.get("final", {}),
        },
    }
    save_json_artifact(payload, BEST_PARAMS_PATH)
    return payload

def export_validation_predictions(
    features,
    targets,
    predictions,
    metadata,
    model_configuration,
    reviewed_validation_mean_km,
    source_notebook,
):
    """Write aligned validation predictions and a provenance sidecar."""
    row_ids = pd.Index(features.index.astype(str), name="ROW_ID")
    if not row_ids.is_unique:
        raise ValueError("Validation prediction export requires unique ROW_ID values.")
    if not features.index.equals(targets.index):
        raise ValueError("Validation features and targets are not index-aligned.")

    truth = np.asarray(targets[TARGET_COLUMNS].to_numpy(), dtype=float)
    prediction_matrix = np.asarray(predictions, dtype=float)
    expected_shape = (len(row_ids), len(TARGET_COLUMNS))
    if truth.shape != expected_shape or prediction_matrix.shape != expected_shape:
        raise ValueError(
            "Validation export shape mismatch: "
            f"truth={truth.shape}, predictions={prediction_matrix.shape}, "
            f"expected={expected_shape}."
        )
    if not np.isfinite(truth).all() or not np.isfinite(prediction_matrix).all():
        raise ValueError("Validation export contains non-finite coordinates.")

    trip_groups = [extract_trip_group(value) for value in row_ids]
    export_frame = pd.DataFrame({
        "ROW_ID": row_ids,
        "TRIP_GROUP": trip_groups,
        "END_LONG": truth[:, 0],
        "END_LAT": truth[:, 1],
        "PRED_END_LONG": prediction_matrix[:, 0],
        "PRED_END_LAT": prediction_matrix[:, 1],
    })
    if export_frame.isna().any().any():
        raise ValueError("Validation prediction export contains missing values.")

    export_metrics, _ = evaluate_long_lat_predictions(
        truth,
        prediction_matrix,
        prefix="export",
    )
    export_mean_km = float(export_metrics["export/haversine_mean_km"])
    reviewed_drift_km = export_mean_km - float(reviewed_validation_mean_km)
    if abs(reviewed_drift_km) > VALIDATION_EXPORT_DRIFT_TOLERANCE_KM:
        raise ValueError(
            f"{MODEL_NAME} validation export drifted by {reviewed_drift_km:.6f} km "
            f"from the reviewed score {reviewed_validation_mean_km:.12f} km."
        )

    export_frame.to_parquet(
        VALIDATION_PREDICTIONS_PATH,
        index=False,
        compression="snappy",
    )
    split_config = metadata.get("config", {})
    manifest = {
        "schema_version": 1,
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "source_notebook": source_notebook,
        "prediction_file": VALIDATION_PREDICTIONS_PATH.name,
        "row_count": int(len(export_frame)),
        "columns": list(export_frame.columns),
        "target_order": TARGET_COLUMNS.copy(),
        "model_configuration": model_configuration,
        "preprocessed_dataset": {
            "schema_version": metadata.get("schema_version"),
            "validation_shape": metadata.get("validation", {}).get("X_val_shape"),
            "config": {
                key: split_config.get(key)
                for key in [
                    "seed",
                    "validation_size",
                    "validation_train_seed",
                    "validation_val_seed",
                    "target_format",
                ]
            },
        },
        "validation_metrics": {
            "mean_haversine_km": export_mean_km,
            "reviewed_mean_haversine_km": float(reviewed_validation_mean_km),
            "drift_from_reviewed_km": float(reviewed_drift_km),
        },
        "kaggle": {
            "kernel_run_id": os.environ.get("KAGGLE_KERNEL_RUN_ID"),
            "kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
        },
    }
    save_json_artifact(manifest, VALIDATION_PREDICTIONS_MANIFEST_PATH)
    return manifest


## Define XGBoost Model


In [ ]:
def effective_model_params(candidate_params, n_estimators):
    """Return the exact merged XGBRegressor configuration."""
    return {
        **FIXED_MODEL_PARAMS,
        **candidate_params,
        "n_estimators": int(n_estimators),
    }


def build_model(candidate_params, n_estimators, callbacks=None):
    """Build one CUDA XGBoost regressor."""
    params = effective_model_params(candidate_params, n_estimators)
    if callbacks is not None:
        params["callbacks"] = callbacks
    return XGBRegressor(**params)


def fit_pair_with_fixed_rounds(features, targets, model_params_by_target, target_rounds):
    """Fit a complete target pair with reviewed parameters and fixed rounds."""
    models = {}
    target_results = {}
    failed_target = None
    try:
        for target_column, target_label in TARGET_SPECS:
            failed_target = target_column
            model_params = model_params_by_target[target_column]
            selected_n_estimators = int(target_rounds[target_column])
            model = build_model(
                candidate_params=model_params,
                n_estimators=selected_n_estimators,
                callbacks=None,
            )
            fit_started = time.time()
            model.fit(
                features,
                targets[target_column].to_numpy(dtype=np.float32),
                verbose=False,
            )
            fit_seconds = float(time.time() - fit_started)
            models[target_column] = model
            target_results[target_column] = {
                "target_column": target_column,
                "target_label": target_label,
                "completed": True,
                "candidate_params": model_params.copy(),
                "selected_n_estimators": selected_n_estimators,
                "fit_seconds": fit_seconds,
                "effective_model_params": effective_model_params(
                    model_params,
                    selected_n_estimators,
                ),
            }
        return models, target_results
    except Exception as exception:
        models.clear()
        gc.collect()
        raise RuntimeError(
            "The fixed-round estimator pair is incomplete; "
            f"target {failed_target!r} did not finish successfully."
        ) from exception


def serialize_xgboost_model_to_ubj(model):
    """Return a stable UBJ model payload without pickling estimator internals."""
    return bytes(model.get_booster().save_raw(raw_format="ubj"))


def build_final_model_bundle(
    model_payloads,
    target_results,
    gps_scaler,
    categorical_encoder,
    feature_columns,
    validation_metrics,
    final_target_n_estimators,
    final_round_calibration,
    execution_status,
):
    '''Build the stable target-specific final model bundle.'''
    final_rounds = {
        target: int(rounds)
        for target, rounds in final_target_n_estimators.items()
    }
    return {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "model_serialized": True,
        "model_format": "ubj_bytes",
        "model_payloads": model_payloads,
        "target_to_payload": TARGET_TO_PAYLOAD.copy(),
        "target_columns": TARGET_COLUMNS.copy(),
        "target_specs": TARGET_SPECS.copy(),
        "model_params_by_target": deepcopy(BEST_TARGET_MODEL_PARAMS),
        "fixed_model_params": FIXED_MODEL_PARAMS.copy(),
        "target_n_estimators": final_rounds.copy(),
        "reviewed_target_n_estimators": BEST_TARGET_N_ESTIMATORS.copy(),
        "final_target_n_estimators": final_rounds.copy(),
        "final_round_calibration": deepcopy(final_round_calibration),
        "target_results": target_results,
        "gps_scaler": gps_scaler,
        "categorical_encoder": categorical_encoder,
        "feature_columns": list(feature_columns),
        "validation_metrics": validation_metrics.copy(),
        "selected_best_run_summary": deepcopy(BEST_MODEL_RUN_SUMMARY),
        "execution": deepcopy(execution_status),
        "final_refit": True,
        "prediction_format": "[LONG, LAT]",
        "preprocessed_data_dir": str(PREPROCESSED_DATA_DIR),
        "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
        "preprocessed_metadata": PREPROCESSED_METADATA,
        "xgboost_runtime": XGBOOST_RUNTIME,
    }


## Hyperparameter Tuning

Use one grouped inner split only to select target-specific round counts.
Refit every candidate on all validation-training rows, then select
longitude/latitude sources by Haversine distance on `validation/X_val`.
Results remain unreviewed until manually promoted into the constants above.


### Search Catalogue


In [ ]:
BALANCED_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 50,
    "subsample": 0.85,
    "colsample_bytree": 0.70,
    "reg_lambda": 5,
    "reg_alpha": 0.0,
}
FLEXIBLE_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 10,
    "subsample": 0.90,
    "colsample_bytree": 0.80,
    "reg_lambda": 1,
    "reg_alpha": 0.0,
}


def candidate(candidate_index, candidate_label, base_params, **overrides):
    """Return one catalogue entry from a base configuration and overrides."""
    return {
        "candidate_index": candidate_index,
        "candidate_label": candidate_label,
        "params": {**base_params, **overrides},
    }


TUNING_CANDIDATES = [
    candidate(1, "xgb_01_balanced_baseline", BALANCED_PARAMS),
    candidate(2, "xgb_02_shallow", BALANCED_PARAMS, max_depth=6),
    candidate(
        3,
        "xgb_03_deep_regularized",
        BALANCED_PARAMS,
        max_depth=10,
        min_child_weight=100,
        reg_lambda=10,
    ),
    candidate(4, "xgb_04_slow_learning", BALANCED_PARAMS, learning_rate=0.03),
    candidate(5, "xgb_05_fast_learning", BALANCED_PARAMS, learning_rate=0.08),
    candidate(6, "xgb_06_flexible", FLEXIBLE_PARAMS),
    candidate(
        7,
        "xgb_07_conservative",
        BALANCED_PARAMS,
        min_child_weight=200,
        subsample=0.80,
        colsample_bytree=0.60,
        reg_lambda=20,
        reg_alpha=0.05,
    ),
    candidate(
        8,
        "xgb_08_strong_sampling",
        BALANCED_PARAMS,
        subsample=0.70,
        colsample_bytree=0.50,
    ),
    candidate(
        9,
        "xgb_09_full_rows_features",
        BALANCED_PARAMS,
        subsample=1.00,
        colsample_bytree=1.00,
    ),
    candidate(
        10,
        "xgb_10_flexible_full_rows_features",
        FLEXIBLE_PARAMS,
        subsample=1.00,
        colsample_bytree=1.00,
    ),
    candidate(11, "xgb_11_deep_flexible", FLEXIBLE_PARAMS, max_depth=10),
    candidate(
        12,
        "xgb_12_moderated_flexible",
        FLEXIBLE_PARAMS,
        min_child_weight=25,
        reg_lambda=2.5,
    ),
    candidate(
        13,
        "xgb_13_boundary_flexible",
        FLEXIBLE_PARAMS,
        min_child_weight=5,
        reg_lambda=0.5,
    ),
    candidate(14, "xgb_14_depth9_flexible", FLEXIBLE_PARAMS, max_depth=9),
    candidate(
        15,
        "xgb_15_deep_moderated",
        FLEXIBLE_PARAMS,
        max_depth=10,
        min_child_weight=25,
        reg_lambda=2.5,
    ),
    candidate(
        16,
        "xgb_16_very_deep_moderated",
        FLEXIBLE_PARAMS,
        max_depth=12,
        min_child_weight=50,
        reg_lambda=5,
    ),
    candidate(17, "xgb_17_slow_flexible", FLEXIBLE_PARAMS, learning_rate=0.03),
    candidate(18, "xgb_18_fast_flexible", FLEXIBLE_PARAMS, learning_rate=0.08),
    candidate(
        19,
        "xgb_19_high_coverage_flexible",
        FLEXIBLE_PARAMS,
        subsample=0.95,
        colsample_bytree=0.90,
    ),
    candidate(
        20,
        "xgb_20_lossguide_128",
        FLEXIBLE_PARAMS,
        max_depth=0,
        grow_policy="lossguide",
        max_leaves=128,
    ),
    candidate(
        21,
        "xgb_21_lossguide_256",
        FLEXIBLE_PARAMS,
        max_depth=0,
        grow_policy="lossguide",
        max_leaves=256,
    ),
    candidate(
        22,
        "xgb_22_gradient_sampling_050",
        FLEXIBLE_PARAMS,
        sampling_method="gradient_based",
        subsample=0.50,
    ),
    candidate(
        23,
        "xgb_23_gradient_sampling_030",
        FLEXIBLE_PARAMS,
        sampling_method="gradient_based",
        subsample=0.30,
    ),
    candidate(26, "xgb_26_max_bin_512_flexible", FLEXIBLE_PARAMS, max_bin=512),
    candidate(
        27,
        "micro_02_depth12_less_regularized",
        FLEXIBLE_PARAMS,
        max_depth=12,
        min_child_weight=25,
        reg_lambda=2.5,
    ),
    candidate(
        28,
        "micro_03_depth12_more_regularized",
        FLEXIBLE_PARAMS,
        max_depth=12,
        min_child_weight=100,
        reg_lambda=10,
    ),
    candidate(
        29,
        "micro_04_depth14_less_regularized",
        FLEXIBLE_PARAMS,
        max_depth=14,
        min_child_weight=25,
        reg_lambda=2.5,
    ),
    candidate(
        30,
        "micro_05_depth14_moderated",
        FLEXIBLE_PARAMS,
        max_depth=14,
        min_child_weight=50,
        reg_lambda=5.0,
    ),
    candidate(
        31,
        "micro_06_depth14_regularized",
        FLEXIBLE_PARAMS,
        max_depth=14,
        min_child_weight=100,
        reg_lambda=10.0,
    ),
    candidate(
        32,
        "micro_08_lossguide256_moderated",
        FLEXIBLE_PARAMS,
        max_depth=0,
        grow_policy="lossguide",
        max_leaves=256,
        min_child_weight=25,
        reg_lambda=2.5,
    ),
    candidate(
        33,
        "micro_09_lossguide384_regularized",
        FLEXIBLE_PARAMS,
        max_depth=0,
        grow_policy="lossguide",
        max_leaves=384,
        min_child_weight=50,
        reg_lambda=5.0,
    ),
    candidate(
        34,
        "micro_10_lossguide512_strong",
        FLEXIBLE_PARAMS,
        max_depth=0,
        grow_policy="lossguide",
        max_leaves=512,
        min_child_weight=100,
        reg_lambda=10.0,
    ),
]


def canonical_parameter_signature(params):
    """Return a numeric-type-insensitive signature for a flat parameter map."""
    normalized = {}
    for key, value in params.items():
        if isinstance(value, bool) or value is None or isinstance(value, str):
            normalized[key] = value
        elif isinstance(value, (int, float, np.integer, np.floating)):
            normalized[key] = float(value)
        else:
            normalized[key] = value
    return json.dumps(normalized, sort_keys=True, separators=(",", ":"))


TUNING_UNIQUE_CANDIDATE_COUNT = len(TUNING_CANDIDATES)
if RUN_HYPERPARAMETER_TUNING:
    if TUNING_UNIQUE_CANDIDATE_COUNT != 32:
        raise RuntimeError("The XGBoost search must contain 32 candidates.")
    expected_candidate_indices = list(range(1, 24)) + list(range(26, 35))
    if [item["candidate_index"] for item in TUNING_CANDIDATES] != expected_candidate_indices:
        raise RuntimeError("Candidate indices must preserve the reviewed catalogue without 24 and 25.")
    if len({item["candidate_label"] for item in TUNING_CANDIDATES}) != 32:
        raise RuntimeError("Candidate labels must be unique.")
    if len({canonical_parameter_signature(item["params"]) for item in TUNING_CANDIDATES}) != 32:
        raise RuntimeError("Candidate parameter configurations must be numerically unique.")


### Tuning Logging Helpers

Validate W&B once before loading the large search matrices. Logging failures
never control model training.


In [ ]:
WANDB_ENABLED = False
_WANDB_MODULE = None
_WANDB_MODE = "disabled"
_WANDB_WARNING_STAGES = set()


def get_wandb_key():
    """Return the W&B API key from Kaggle secrets or the environment."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    except Exception:
        return os.environ.get("WANDB_API_KEY")


def wandb_warning(stage, exception):
    """Emit one sanitized warning per W&B stage."""
    if stage not in _WANDB_WARNING_STAGES:
        print(
            f"W&B {stage} failed ({type(exception).__name__}). "
            "Model tuning will continue."
        )
        _WANDB_WARNING_STAGES.add(stage)


def initialize_wandb():
    """Validate W&B credentials once before optional tuning starts."""
    wandb_key = get_wandb_key()
    if not wandb_key:
        print("No W&B key found. Tuning will run without W&B logging.")
        return None, "disabled", False
    try:
        import wandb

        login_succeeded = wandb.login(key=wandb_key)
        if login_succeeded is False:
            raise RuntimeError("W&B login was rejected.")
        return wandb, "online", True
    except Exception as exception:
        wandb_warning("setup", exception)
        return None, "disabled", False


if RUN_HYPERPARAMETER_TUNING:
    _WANDB_MODULE, _WANDB_MODE, WANDB_ENABLED = initialize_wandb()


WANDB_TARGET_METRICS = {
    "longitude": {
        "round": "longitude/boosting_round",
        "train": "longitude/inner_train_rmse",
        "validation": "longitude/inner_stop_rmse",
    },
    "latitude": {
        "round": "latitude/boosting_round",
        "train": "latitude/inner_train_rmse",
        "validation": "latitude/inner_stop_rmse",
    },
}


class TargetWandbCallback(xgb.callback.TrainingCallback):
    """Log one target history while always allowing training to continue."""

    def __init__(self, run, target_label):
        self.run = run
        self.target_label = target_label
        self.disabled = run is None

    @staticmethod
    def latest_metric(evals_log, dataset_name, metric_name):
        values = evals_log.get(dataset_name, {}).get(metric_name, [])
        if not values:
            return None
        value = values[-1]
        if isinstance(value, tuple):
            value = value[0]
        return float(value)

    def after_iteration(self, model, epoch, evals_log):
        if self.disabled:
            return False
        try:
            names = WANDB_TARGET_METRICS[self.target_label]
            payload = {names["round"]: int(epoch)}
            train_value = self.latest_metric(evals_log, "validation_0", "rmse")
            stop_value = self.latest_metric(evals_log, "validation_1", "rmse")
            if train_value is not None:
                payload[names["train"]] = train_value
            if stop_value is not None:
                payload[names["validation"]] = stop_value
            self.run.log(payload)
        except Exception as exception:
            wandb_warning(f"{self.target_label} logging", exception)
            self.disabled = True
        return False


def initialize_candidate_wandb_run(candidate):
    """Start one optional W&B run for a complete target pair."""
    if not WANDB_ENABLED:
        return None
    run = None
    try:
        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=f"xgboost-search-{candidate['candidate_label']}",
            job_type="xgboost_candidate_pair",
            mode=_WANDB_MODE,
            config={
                "model_slug": MODEL_SLUG,
                "trial_index": candidate["candidate_index"],
                "trial_params": candidate["params"],
                "seed": SEED,
                "tuner_type": "canonical_search",
                "candidate_index": candidate["candidate_index"],
                "candidate_label": candidate["candidate_label"],
                "candidate_params": candidate["params"],
                "fixed_model_params": FIXED_MODEL_PARAMS,
                "max_boost_rounds": MAX_BOOST_ROUNDS,
                "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
                "search_stop_fraction": SEARCH_STOP_FRACTION,
            },
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no candidate run.")
        for names in WANDB_TARGET_METRICS.values():
            run.define_metric(names["round"])
            run.define_metric(
                names["train"],
                step_metric=names["round"],
                summary="min",
            )
            run.define_metric(
                names["validation"],
                step_metric=names["round"],
                summary="min",
            )
        return run
    except Exception as exception:
        wandb_warning("candidate setup", exception)
        if run is not None:
            try:
                run.finish()
            except Exception as finish_exception:
                wandb_warning("candidate setup finalization", finish_exception)
        return None


def safe_wandb_summary(run, payload):
    """Write optional summary values without affecting search."""
    if run is None:
        return
    try:
        for key, value in make_json_safe(payload).items():
            run.summary[key] = value
    except Exception as exception:
        wandb_warning("summary", exception)


def safe_wandb_log(run, payload):
    """Log optional values without affecting search."""
    if run is None:
        return
    try:
        run.log(payload)
    except Exception as exception:
        wandb_warning("logging", exception)


def safe_wandb_finish(run):
    """Finish an optional run without masking model results."""
    if run is None:
        return
    try:
        run.finish()
    except Exception as exception:
        wandb_warning("finalization", exception)


def log_search_summary_to_wandb(candidate_results, hybrid_results, winner):
    """Log in-memory search tables and the unreviewed winner."""
    if not WANDB_ENABLED:
        return
    run = None
    try:
        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name="xgboost-search-summary",
            job_type="xgboost_search_summary",
            mode=_WANDB_MODE,
            config={
                "model_slug": MODEL_SLUG,
                "seed": SEED,
                "tuner_type": "canonical_summary",
                "candidate_count": len(candidate_results),
                "hybrid_count": len(hybrid_results),
            },
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no summary run.")
        run.log({
            "candidate_results": _WANDB_MODULE.Table(dataframe=candidate_results),
            "hybrid_results": _WANDB_MODULE.Table(dataframe=hybrid_results),
        })
        safe_wandb_summary(run, {
            "winning_longitude_candidate": winner["longitude_candidate_label"],
            "winning_latitude_candidate": winner["latitude_candidate_label"],
            "validation_haversine_mean_km": winner["haversine_mean_km"],
            "validation_haversine_p90_km": winner["haversine_p90_km"],
            "hybrid_count": len(hybrid_results),
            "promotion_status": "requires_manual_review",
        })
    except Exception as exception:
        wandb_warning("search summary", exception)
    finally:
        safe_wandb_finish(run)


### Candidate Evaluation


In [ ]:
def build_target_callbacks(run, target_label):
    '''Create fresh early-stopping and optional W&B callbacks.'''
    callbacks = [
        xgb.callback.EarlyStopping(
            rounds=EARLY_STOPPING_ROUNDS,
            metric_name="rmse",
            data_name="validation_1",
            maximize=False,
            save_best=True,
        )
    ]
    if run is not None:
        callbacks.append(TargetWandbCallback(run, target_label))
    return callbacks


def fit_target_with_early_stopping(
    model_params,
    target_column,
    target_label,
    run,
    X_fit,
    y_fit,
    X_stop,
    y_stop,
):
    '''Select one target's boosting rounds on a group-disjoint stop set.'''
    model = build_model(
        candidate_params=model_params,
        n_estimators=MAX_BOOST_ROUNDS,
        callbacks=build_target_callbacks(run, target_label),
    )
    y_fit_target = y_fit[target_column].to_numpy(dtype=np.float32)
    y_stop_target = y_stop[target_column].to_numpy(dtype=np.float32)
    try:
        fit_started = time.time()
        model.fit(
            X_fit,
            y_fit_target,
            eval_set=[
                (X_fit, y_fit_target),
                (X_stop, y_stop_target),
            ],
            verbose=False,
        )
        fit_seconds = float(time.time() - fit_started)

        best_iteration = int(model.best_iteration)
        selected_n_estimators = best_iteration + 1
        evaluation_history = model.evals_result()
        trained_rounds = len(evaluation_history["validation_1"]["rmse"])
        patience_observed = trained_rounds - selected_n_estimators
        ceiling_reached = trained_rounds >= MAX_BOOST_ROUNDS
        truncated_at_ceiling = bool(
            ceiling_reached and patience_observed < EARLY_STOPPING_ROUNDS
        )
        completion_state = (
            "truncated_at_ceiling" if truncated_at_ceiling else "completed"
        )
        return {
            "target_column": target_column,
            "target_label": target_label,
            "completed": True,
            "completion_state": completion_state,
            "eligible_for_hybrid": not truncated_at_ceiling,
            "candidate_params": model_params.copy(),
            "best_iteration": best_iteration,
            "selected_n_estimators": selected_n_estimators,
            "trained_rounds": trained_rounds,
            "patience_rounds_observed": patience_observed,
            "best_inner_stop_rmse": float(model.best_score),
            "early_stopping_fit_seconds": fit_seconds,
            "ceiling_reached": ceiling_reached,
            "effective_model_params": effective_model_params(
                model_params,
                selected_n_estimators,
            ),
        }
    finally:
        del model
        gc.collect()


def fit_target_on_validation_train(
    model_params,
    target_column,
    selected_n_estimators,
    X_train,
    y_train,
    X_val,
    y_val,
):
    '''Refit one target on all X_train rows and predict X_val.'''
    model = build_model(
        candidate_params=model_params,
        n_estimators=selected_n_estimators,
        callbacks=None,
    )
    try:
        fit_started = time.time()
        model.fit(
            X_train,
            y_train[target_column].to_numpy(dtype=np.float32),
            verbose=False,
        )
        fit_seconds = float(time.time() - fit_started)
        prediction = np.asarray(model.predict(X_val), dtype=np.float32)
        result = {
            "validation_refit_completed": True,
            "validation_refit_completion_state": "completed",
            "validation_refit_fit_seconds": fit_seconds,
            "validation_rmse": rmse(
                y_val[target_column].to_numpy(dtype=np.float32),
                prediction,
            ),
        }
        return result, prediction
    finally:
        del model
        gc.collect()


def evaluate_candidate_pair(
    candidate,
    X_inner_train,
    y_inner_train,
    X_inner_stop,
    y_inner_stop,
    X_train,
    y_train,
    X_val,
    y_val,
):
    '''Select rounds, refit both targets, and score one candidate on X_val.'''
    run = initialize_candidate_wandb_run(candidate)
    pair_started = time.time()
    target_results = {}
    validation_predictions = {}
    failed_target = None
    try:
        for target_column, target_label in TARGET_SPECS:
            failed_target = target_column
            target_result = fit_target_with_early_stopping(
                candidate["params"],
                target_column,
                target_label,
                run,
                X_inner_train,
                y_inner_train,
                X_inner_stop,
                y_inner_stop,
            )
            target_result.update({
                "validation_refit_completed": False,
                "validation_refit_completion_state": "not_run_truncated",
                "validation_refit_fit_seconds": None,
                "validation_rmse": None,
            })
            if target_result["eligible_for_hybrid"]:
                refit_result, validation_prediction = (
                    fit_target_on_validation_train(
                        candidate["params"],
                        target_column,
                        target_result["selected_n_estimators"],
                        X_train,
                        y_train,
                        X_val,
                        y_val,
                    )
                )
                target_result.update(refit_result)
                validation_predictions[target_column] = validation_prediction
            target_results[target_column] = target_result

        eligible_target_count = sum(
            bool(result["eligible_for_hybrid"])
            and bool(result["validation_refit_completed"])
            for result in target_results.values()
        )
        pair_completion_state = {
            2: "completed",
            1: "partially_eligible",
            0: "no_eligible_targets",
        }[eligible_target_count]
        validation_metrics = {}
        if eligible_target_count == len(TARGET_COLUMNS):
            combined_validation = combine_target_predictions(
                validation_predictions
            )
            validation_metrics, _ = evaluate_long_lat_predictions(
                y_val[TARGET_COLUMNS].to_numpy(),
                combined_validation,
                "validation",
            )

        record = {
            "candidate_index": candidate["candidate_index"],
            "candidate_label": candidate["candidate_label"],
            "candidate_params": candidate["params"],
            "pair_completion_state": pair_completion_state,
            "target_results": target_results,
            "validation_metrics": validation_metrics,
            "early_stopping_fit_seconds": float(sum(
                result["early_stopping_fit_seconds"]
                for result in target_results.values()
            )),
            "validation_refit_fit_seconds": float(sum(
                result["validation_refit_fit_seconds"] or 0.0
                for result in target_results.values()
            )),
            "elapsed_seconds": float(time.time() - pair_started),
        }
        standardized_metrics = {}
        if validation_metrics:
            standardized_metrics = {
                "val/haversine_mean_km": validation_metrics[
                    "validation/haversine_mean_km"
                ],
                "val/haversine_median_km": validation_metrics[
                    "validation/haversine_median_km"
                ],
                "val/haversine_std_km": validation_metrics[
                    "validation/haversine_std_km"
                ],
                "val/haversine_p90_km": validation_metrics[
                    "validation/haversine_p90_km"
                ],
                "fit_seconds": record["validation_refit_fit_seconds"],
            }
        summary = {
            "execution_completed": True,
            "pair_completion_state": pair_completion_state,
            "early_stopping_fit_seconds": record[
                "early_stopping_fit_seconds"
            ],
            "validation_refit_fit_seconds": record[
                "validation_refit_fit_seconds"
            ],
            "elapsed_seconds": record["elapsed_seconds"],
            **validation_metrics,
            **standardized_metrics,
        }
        for target_column, target_label in TARGET_SPECS:
            target_result = target_results[target_column]
            summary.update({
                f"{target_label}_best_iteration": target_result[
                    "best_iteration"
                ],
                f"{target_label}_selected_rounds": target_result[
                    "selected_n_estimators"
                ],
                f"{target_label}_best_inner_stop_rmse": target_result[
                    "best_inner_stop_rmse"
                ],
                f"{target_label}_early_stopping_fit_seconds": target_result[
                    "early_stopping_fit_seconds"
                ],
                f"{target_label}_validation_refit_fit_seconds": target_result[
                    "validation_refit_fit_seconds"
                ],
                f"{target_label}_validation_rmse": target_result[
                    "validation_rmse"
                ],
                f"{target_label}_completion_state": target_result[
                    "completion_state"
                ],
            })
        safe_wandb_summary(run, summary)
        if validation_metrics:
            safe_wandb_log(run, {
                **validation_metrics,
                **standardized_metrics,
            })
        return record, validation_predictions
    except Exception as exception:
        safe_wandb_summary(run, {
            "execution_completed": False,
            "failed_target": failed_target,
            "failure_type": type(exception).__name__,
        })
        raise RuntimeError(
            f"Candidate {candidate['candidate_label']} is incomplete; "
            f"target {failed_target!r} failed."
        ) from exception
    finally:
        safe_wandb_finish(run)
        gc.collect()


def candidate_record_to_row(candidate, record):
    '''Flatten one candidate record for display and W&B.'''
    row = {
        "candidate_index": candidate["candidate_index"],
        "candidate_label": candidate["candidate_label"],
        "pair_completion_state": record.get(
            "pair_completion_state",
            "missing",
        ),
        "candidate_params": repr(candidate["params"]),
    }
    target_results = record.get("target_results", {})
    for target_column, target_label in TARGET_SPECS:
        result = target_results.get(target_column, {})
        row.update({
            f"{target_label}_completion_state": result.get(
                "completion_state"
            ),
            f"{target_label}_eligible_for_hybrid": result.get(
                "eligible_for_hybrid",
                False,
            ),
            f"{target_label}_best_iteration": result.get(
                "best_iteration"
            ),
            f"{target_label}_selected_rounds": result.get(
                "selected_n_estimators"
            ),
            f"{target_label}_best_inner_stop_rmse": result.get(
                "best_inner_stop_rmse"
            ),
            f"{target_label}_validation_rmse": result.get(
                "validation_rmse"
            ),
            f"{target_label}_early_stopping_fit_seconds": result.get(
                "early_stopping_fit_seconds"
            ),
            f"{target_label}_validation_refit_fit_seconds": result.get(
                "validation_refit_fit_seconds"
            ),
            f"{target_label}_ceiling_reached": result.get(
                "ceiling_reached"
            ),
        })
    row.update(record.get("validation_metrics", {}))
    row["early_stopping_fit_seconds"] = record.get(
        "early_stopping_fit_seconds"
    )
    row["validation_refit_fit_seconds"] = record.get(
        "validation_refit_fit_seconds"
    )
    row["elapsed_seconds"] = record.get("elapsed_seconds")
    return row


### Hybrid Selection


In [ ]:
HYBRID_SORT_COLUMNS = [
    "haversine_mean_km",
    "haversine_p90_km",
    "haversine_median_km",
    "total_selected_rounds",
    "longitude_candidate_label",
    "latitude_candidate_label",
]


def eligible_target_labels(candidate_catalog, records, target_column):
    '''Return catalogue-ordered labels with complete X_val predictions.'''
    labels = []
    for candidate_item in candidate_catalog:
        record = records.get(candidate_item["candidate_label"], {})
        target = record.get("target_results", {}).get(target_column, {})
        if (
            target.get("completed") is True
            and target.get("eligible_for_hybrid") is True
            and target.get("completion_state") == "completed"
            and target.get("validation_refit_completed") is True
            and target.get("validation_refit_completion_state") == "completed"
        ):
            labels.append(candidate_item["candidate_label"])
    return labels


def enumerate_hybrid_rows(
    longitude_labels,
    latitude_labels,
    predictions_by_candidate,
    records,
    truth_long_lat,
):
    '''Evaluate every eligible hybrid on the held-out X_val rows.'''
    rows = []
    for longitude_label in longitude_labels:
        longitude_prediction = predictions_by_candidate[
            longitude_label
        ]["END_LONG"]
        longitude_result = records[longitude_label]["target_results"][
            "END_LONG"
        ]
        for latitude_label in latitude_labels:
            latitude_prediction = predictions_by_candidate[
                latitude_label
            ]["END_LAT"]
            latitude_result = records[latitude_label]["target_results"][
                "END_LAT"
            ]
            combined = combine_target_predictions({
                "END_LONG": longitude_prediction,
                "END_LAT": latitude_prediction,
            })
            metrics, _ = evaluate_long_lat_predictions(
                truth_long_lat,
                combined,
                "validation",
            )
            rows.append({
                "longitude_candidate_label": longitude_label,
                "latitude_candidate_label": latitude_label,
                "longitude_selected_rounds": longitude_result[
                    "selected_n_estimators"
                ],
                "latitude_selected_rounds": latitude_result[
                    "selected_n_estimators"
                ],
                "total_selected_rounds": int(
                    longitude_result["selected_n_estimators"]
                    + latitude_result["selected_n_estimators"]
                ),
                "haversine_mean_km": metrics[
                    "validation/haversine_mean_km"
                ],
                "haversine_median_km": metrics[
                    "validation/haversine_median_km"
                ],
                "haversine_std_km": metrics[
                    "validation/haversine_std_km"
                ],
                "haversine_p90_km": metrics[
                    "validation/haversine_p90_km"
                ],
            })
    return rows


def deterministically_rank_hybrids(hybrid_rows):
    """Sort hybrid evidence using the complete deterministic contract."""
    result = pd.DataFrame(hybrid_rows)
    if result.empty:
        raise RuntimeError("No eligible hybrid was generated.")
    result = result.sort_values(
        HYBRID_SORT_COLUMNS,
        ascending=True,
        kind="mergesort",
    ).reset_index(drop=True)
    result.insert(0, "hybrid_rank", np.arange(1, len(result) + 1))
    return result


### Run Hyperparameter Tuning


In [ ]:
def run_hyperparameter_tuning():
    '''Select rounds internally, then rank every hybrid on X_val.'''
    if not RUN_HYPERPARAMETER_TUNING:
        print("Hyperparameter tuning disabled. Using reviewed parameters.")
        return pd.DataFrame(), pd.DataFrame(), None

    X_train, y_train, X_val, y_val = load_cached_validation_data()
    train_positions, stop_positions, trip_groups = make_group_disjoint_split(
        X_train.index,
        stop_fraction=SEARCH_STOP_FRACTION,
        seed=SEED,
    )
    X_inner_train = X_train.iloc[train_positions].copy()
    y_inner_train = y_train.iloc[train_positions].copy()
    X_inner_stop = X_train.iloc[stop_positions].copy()
    y_inner_stop = y_train.iloc[stop_positions].copy()

    train_group_count = len(set(trip_groups[train_positions]))
    stop_group_count = len(set(trip_groups[stop_positions]))
    del trip_groups, train_positions, stop_positions
    gc.collect()

    display(pd.Series({
        "validation_train_rows": len(X_train),
        "inner_train_rows": len(X_inner_train),
        "inner_stop_rows": len(X_inner_stop),
        "validation_holdout_rows": len(X_val),
        "inner_train_trip_groups": train_group_count,
        "inner_stop_trip_groups": stop_group_count,
        "overlapping_trip_groups": 0,
    }))

    candidate_records = {}
    candidate_predictions = {}
    for candidate_position, candidate_item in enumerate(
        TUNING_CANDIDATES,
        start=1,
    ):
        record, predictions = evaluate_candidate_pair(
            candidate_item,
            X_inner_train,
            y_inner_train,
            X_inner_stop,
            y_inner_stop,
            X_train,
            y_train,
            X_val,
            y_val,
        )
        label = candidate_item["candidate_label"]
        candidate_records[label] = record
        candidate_predictions[label] = predictions
        validation_mean = record.get("validation_metrics", {}).get(
            "validation/haversine_mean_km"
        )
        result_text = (
            f"{validation_mean:.6f} km"
            if validation_mean is not None
            else record["pair_completion_state"]
        )
        print(
            f"[{candidate_position:02d}/{TUNING_UNIQUE_CANDIDATE_COUNT}] "
            f"candidate {candidate_item['candidate_index']:02d} "
            f"{label}: {result_text}"
        )

    candidate_results = pd.DataFrame([
        candidate_record_to_row(
            candidate_item,
            candidate_records[candidate_item["candidate_label"]],
        )
        for candidate_item in TUNING_CANDIDATES
    ]).sort_values("candidate_index").reset_index(drop=True)

    longitude_labels = eligible_target_labels(
        TUNING_CANDIDATES,
        candidate_records,
        "END_LONG",
    )
    latitude_labels = eligible_target_labels(
        TUNING_CANDIDATES,
        candidate_records,
        "END_LAT",
    )
    hybrid_rows = enumerate_hybrid_rows(
        longitude_labels,
        latitude_labels,
        candidate_predictions,
        candidate_records,
        y_val[TARGET_COLUMNS].to_numpy(),
    )
    expected_hybrid_count = len(longitude_labels) * len(latitude_labels)
    if len(hybrid_rows) != expected_hybrid_count:
        raise AssertionError(
            "Hybrid count differs from the eligible Cartesian product."
        )
    if (
        len(longitude_labels) == TUNING_UNIQUE_CANDIDATE_COUNT
        and len(latitude_labels) == TUNING_UNIQUE_CANDIDATE_COUNT
        and expected_hybrid_count != 1_024
    ):
        raise AssertionError("A complete search must create 1,024 hybrids.")

    hybrid_results = deterministically_rank_hybrids(hybrid_rows)
    winning_hybrid = hybrid_results.iloc[0].to_dict()
    candidate_by_label = {
        item["candidate_label"]: item
        for item in TUNING_CANDIDATES
    }
    unreviewed_winner = {
        "promotion_status": "requires_manual_review",
        "selection_dataset": "validation/X_val",
        "winning_hybrid": winning_hybrid,
        "target_selection": {
            "END_LONG": {
                "candidate_label": winning_hybrid[
                    "longitude_candidate_label"
                ],
                "candidate_params": candidate_by_label[
                    winning_hybrid["longitude_candidate_label"]
                ]["params"].copy(),
                "selected_n_estimators": candidate_records[
                    winning_hybrid["longitude_candidate_label"]
                ]["target_results"]["END_LONG"]["selected_n_estimators"],
            },
            "END_LAT": {
                "candidate_label": winning_hybrid[
                    "latitude_candidate_label"
                ],
                "candidate_params": candidate_by_label[
                    winning_hybrid["latitude_candidate_label"]
                ]["params"].copy(),
                "selected_n_estimators": candidate_records[
                    winning_hybrid["latitude_candidate_label"]
                ]["target_results"]["END_LAT"]["selected_n_estimators"],
            },
        },
    }

    log_search_summary_to_wandb(
        candidate_results,
        hybrid_results,
        winning_hybrid,
    )
    display(candidate_results)
    display(hybrid_results.head(20))
    display(pd.Series({
        "promotion_status": unreviewed_winner["promotion_status"],
        "selection_dataset": unreviewed_winner["selection_dataset"],
        "longitude_candidate": winning_hybrid[
            "longitude_candidate_label"
        ],
        "latitude_candidate": winning_hybrid[
            "latitude_candidate_label"
        ],
        "validation_haversine_mean_km": winning_hybrid[
            "haversine_mean_km"
        ],
        "validation_haversine_p90_km": winning_hybrid[
            "haversine_p90_km"
        ],
    }))

    candidate_predictions.clear()
    candidate_records.clear()
    del hybrid_rows
    del X_inner_train, y_inner_train, X_inner_stop, y_inner_stop
    del X_train, y_train, X_val, y_val
    gc.collect()
    return candidate_results, hybrid_results, unreviewed_winner


tuning_results, hybrid_results, unreviewed_search_winner = (
    run_hyperparameter_tuning()
)


## Train And Validate

After manual promotion, rerun the reviewed target pair on all
`validation/X_train` rows with fixed target-specific rounds. `X_val` is
prediction-only; this mode produces reproducible diagnostics and the
reviewed parameters artifact rather than new model-selection evidence.


In [ ]:
validation_export_manifest = None

validation_metrics = (
    {}
    if RUN_HYPERPARAMETER_TUNING or RUN_VALIDATION_REFIT
    else reviewed_validation_metrics()
)
target_results = deepcopy(BEST_MODEL_RUN_SUMMARY.get("target_results", {}))
validation_distances = None
execution_status = {
    "hyperparameter_tuning_requested": RUN_HYPERPARAMETER_TUNING,
    "hyperparameter_tuning_completed": bool(
        RUN_HYPERPARAMETER_TUNING and not tuning_results.empty
    ),
    "validation_refit_requested": RUN_VALIDATION_REFIT,
    "validation_refit_completed": False,
    "final_fit_requested": RUN_FINAL_FIT,
    "final_round_calibration_requested": RUN_FINAL_FIT,
    "final_round_calibration_completed": False,
    "final_fit_completed": False,
    "model_bundle_written": False,
    "model_serialized": False,
    "submission_written": False,
    "validation_predictions_written": False,
}

if RUN_VALIDATION_REFIT:
    X_train, y_train, X_val, y_val = load_cached_validation_data()
    validation_models, target_results = fit_pair_with_fixed_rounds(
        X_train,
        y_train,
        BEST_TARGET_MODEL_PARAMS,
        BEST_TARGET_N_ESTIMATORS,
    )
    train_predictions = combine_target_predictions({
        target_column: validation_models[target_column].predict(X_train)
        for target_column in TARGET_COLUMNS
    })
    val_predictions = combine_target_predictions({
        target_column: validation_models[target_column].predict(X_val)
        for target_column in TARGET_COLUMNS
    })
    train_metrics, _ = evaluate_long_lat_predictions(
        y_train[TARGET_COLUMNS].to_numpy(),
        train_predictions,
        "train",
    )
    val_metrics, validation_distances = evaluate_long_lat_predictions(
        y_val[TARGET_COLUMNS].to_numpy(),
        val_predictions,
        "val",
    )
    validation_metrics = {
        **train_metrics,
        **val_metrics,
        "generalization/haversine_gap_km": float(
            val_metrics["val/haversine_mean_km"]
            - train_metrics["train/haversine_mean_km"]
        ),
        "fit_seconds": float(
            sum(result["fit_seconds"] for result in target_results.values())
        ),
        "validation_metrics_source": "selected_model_validation_refit",
    }
    validation_export_manifest = export_validation_predictions(
        features=X_val,
        targets=y_val,
        predictions=val_predictions,
        metadata=PREPROCESSED_METADATA,
        model_configuration={
            "target_model_params": deepcopy(BEST_TARGET_MODEL_PARAMS),
            "target_n_estimators": BEST_TARGET_N_ESTIMATORS.copy(),
            "fixed_model_params": FIXED_MODEL_PARAMS.copy(),
        },
        reviewed_validation_mean_km=BEST_MODEL_RUN_SUMMARY["validation_haversine_mean_km"],
        source_notebook="competition_xgboost.ipynb",
    )
    execution_status.update({
        "validation_refit_completed": True,
        "validation_predictions_written": True,
        "validation_predictions_path": str(VALIDATION_PREDICTIONS_PATH),
        "validation_predictions_manifest_path": str(VALIDATION_PREDICTIONS_MANIFEST_PATH),
        "validation_prediction_rows": validation_export_manifest["row_count"],
    })
    save_best_params_artifact(
        validation_metrics,
        target_results,
        execution_status,
    )
    validation_models.clear()
    gc.collect()
    display(pd.Series({
        key: value
        for key, value in validation_metrics.items()
        if not key.endswith("seconds")
    }))
elif not RUN_HYPERPARAMETER_TUNING:
    print("Using reviewed validation metrics from BEST_MODEL_RUN_SUMMARY.")
    display(pd.Series(validation_metrics))


## Train vs Validation Plot

Display only the standard mean-Haversine comparison and validation-error histogram.


In [ ]:
if RUN_VALIDATION_REFIT:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    mean_values = [
        validation_metrics["train/haversine_mean_km"],
        validation_metrics["val/haversine_mean_km"],
    ]
    bars = axes[0].bar(
        ["Train", "Validation"],
        mean_values,
        color=["#4C78A8", "#F58518"],
    )
    axes[0].set_ylabel("Mean Haversine distance (km)")
    axes[0].set_title(f"{MODEL_NAME} train vs validation error")
    axes[0].bar_label(bars, fmt="%.3f km")

    axes[1].hist(validation_distances, bins=40, color="#F58518")
    axes[1].axvline(
        validation_distances.mean(),
        color="red",
        linestyle="--",
        label=f"Mean: {validation_distances.mean():.3f} km",
    )
    axes[1].set_xlabel("Haversine distance (km)")
    axes[1].set_ylabel("Validation trips")
    axes[1].set_title(f"{MODEL_NAME} validation error distribution")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    for name in [
        "train_predictions",
        "val_predictions",
        "validation_distances",
        "validation_export_manifest",
        "X_train",
        "y_train",
        "X_val",
        "y_val",
    ]:
        globals().pop(name, None)
    gc.collect()
else:
    print("Skipping validation plots because RUN_VALIDATION_REFIT=False.")


## Final Fit And Kaggle Submission

Recalibrate only the reviewed targets' boosting rounds on a temporary
grouped split of the complete final training matrix. Discard those models,
refit fresh boosters on every labeled row, store stable UBJ payloads, and
generate the competition submission.


In [ ]:
if RUN_FINAL_FIT:
    (
        X_final_train,
        y_final_train,
        X_final_test,
        final_gps_scaler,
        final_categorical_encoder,
    ) = load_cached_final_data()

    (
        final_calibration_train_positions,
        final_calibration_stop_positions,
        final_calibration_groups,
    ) = make_group_disjoint_split(
        X_final_train.index,
        stop_fraction=FINAL_ROUND_STOP_FRACTION,
        seed=FINAL_ROUND_CALIBRATION_SEED,
    )
    X_final_calibration_train = X_final_train.iloc[
        final_calibration_train_positions
    ].copy()
    y_final_calibration_train = y_final_train.iloc[
        final_calibration_train_positions
    ].copy()
    X_final_calibration_stop = X_final_train.iloc[
        final_calibration_stop_positions
    ].copy()
    y_final_calibration_stop = y_final_train.iloc[
        final_calibration_stop_positions
    ].copy()

    final_calibration_train_group_count = len(set(
        final_calibration_groups[final_calibration_train_positions]
    ))
    final_calibration_stop_group_count = len(set(
        final_calibration_groups[final_calibration_stop_positions]
    ))
    del final_calibration_groups
    del final_calibration_train_positions, final_calibration_stop_positions
    gc.collect()

    final_round_calibration = {}
    final_target_n_estimators = {}
    for target_column, target_label in TARGET_SPECS:
        calibration_result = fit_target_with_early_stopping(
            BEST_TARGET_MODEL_PARAMS[target_column],
            target_column,
            target_label,
            None,
            X_final_calibration_train,
            y_final_calibration_train,
            X_final_calibration_stop,
            y_final_calibration_stop,
        )
        calibration_result.update({
            "calibration_stop_fraction": FINAL_ROUND_STOP_FRACTION,
            "calibration_seed": FINAL_ROUND_CALIBRATION_SEED,
            "calibration_train_rows": len(X_final_calibration_train),
            "calibration_stop_rows": len(X_final_calibration_stop),
            "calibration_train_trip_groups": (
                final_calibration_train_group_count
            ),
            "calibration_stop_trip_groups": (
                final_calibration_stop_group_count
            ),
            "overlapping_trip_groups": 0,
        })
        final_round_calibration[target_column] = calibration_result
        if calibration_result["completion_state"] != "completed":
            raise RuntimeError(
                f"Final round calibration for {target_column} reached the "
                "10,000-round ceiling without completing early-stopping "
                "patience. Final fit was not started."
            )
        final_target_n_estimators[target_column] = int(
            calibration_result["selected_n_estimators"]
        )

    execution_status["final_round_calibration_completed"] = True
    del X_final_calibration_train, y_final_calibration_train
    del X_final_calibration_stop, y_final_calibration_stop
    gc.collect()

    final_models, final_target_results = fit_pair_with_fixed_rounds(
        X_final_train,
        y_final_train,
        BEST_TARGET_MODEL_PARAMS,
        final_target_n_estimators,
    )
    test_predictions = combine_target_predictions({
        target_column: final_models[target_column].predict(X_final_test)
        for target_column in TARGET_COLUMNS
    })

    submission_path = SUBMISSION_DIR / SUBMISSION_FILE
    submission = write_submission(
        X_final_test.index,
        long_lat_to_lat_long(test_predictions),
        submission_path,
    )
    execution_status["submission_written"] = True
    execution_status["final_fit_completed"] = True

    model_payloads = {
        target_label: {
            "target_column": target_column,
            "format": "ubj",
            "payload": serialize_xgboost_model_to_ubj(
                final_models[target_column]
            ),
        }
        for target_column, target_label in TARGET_SPECS
    }
    bundle_execution_status = {
        **execution_status,
        "model_bundle_written": True,
        "model_serialized": True,
    }
    model_bundle = build_final_model_bundle(
        model_payloads=model_payloads,
        target_results=final_target_results,
        gps_scaler=final_gps_scaler,
        categorical_encoder=final_categorical_encoder,
        feature_columns=X_final_train.columns,
        validation_metrics=validation_metrics,
        final_target_n_estimators=final_target_n_estimators,
        final_round_calibration=final_round_calibration,
        execution_status=bundle_execution_status,
    )
    joblib.dump(model_bundle, MODEL_BUNDLE_PATH)
    execution_status.update({
        "model_bundle_written": True,
        "model_serialized": True,
    })
    save_best_params_artifact(
        validation_metrics,
        final_target_results,
        execution_status,
        final_target_n_estimators=final_target_n_estimators,
        final_round_calibration=final_round_calibration,
    )

    print("Model bundle written to:", MODEL_BUNDLE_PATH.resolve())
    print("Submission written to:", submission_path.resolve())
    display(submission.head())

    final_models.clear()
    gc.collect()
else:
    print("Skipping final fit because RUN_FINAL_FIT=False.")
